# Nested Cross Validation

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, # classificazione
    r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error # regressione
)
from time import time


def nested_cv(model, param_grid, X_train, y_train,
              outer_splits=5, inner_splits=5,
              scoring: list[str] = None,
              random_state=42, verbose=True):

    if verbose:
        start_time = time()

    # Assicurati che `y` sia un array 1D
    if isinstance(y_train, pd.DataFrame):  # Se è un DataFrame Pandas
        y_train = y_train.values.ravel()
    elif isinstance(y_train, pd.Series):  # Se è una Serie Pandas
        y_train = y_train.values
    else:  # Se è un array Numpy
        y_train = np.ravel(y_train)

    # Determina il tipo di task di apprendimento automatico
    est_type = getattr(model, "_estimator_type", None)
    is_clf = (est_type == "classifier")

    if scoring is None:
        if is_clf:
            scoring = ['accuracy', 'roc_auc', 'f1', 'precision', 'recall']
        else:
            scoring = ['r2', 'mae', 'rmse']

    # CROSS-VALIDATION ESTERNA
    outer_cv = KFold(n_splits=outer_splits, shuffle=True, random_state=random_state)

    # Dizionari per salvare i risultati
    score_results = {metric: [] for metric in scoring}
    all_fold_best_params = []

    for outer_fold, (train_fold_idx, val_idx) in enumerate(outer_cv.split(X_train), 1):
        if verbose:
            print(f"\nPerforming Outer Fold {outer_fold}/{outer_splits}")

        # Usare il metodo .iloc per X, se è un DataFrame
        if isinstance(X_train, pd.DataFrame):
            X_train_fold, X_val = X_train.iloc[train_fold_idx], X_train.iloc[val_idx]
        else:  # Altrimenti usa indicizzazione standard
            X_train_fold, X_val = X_train[train_fold_idx], X_train[val_idx]

        y_train_fold, y_val = y_train[train_fold_idx], y_train[val_idx]

        # --- 3. CICLO DI CROSS-VALIDATION INTERNA (TUNING) ---
        inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=random_state)
        primary_metric = scoring[0]  # GridSearchCV ottimizza per la prima metrica della lista

        if verbose:
            print(f"Performing GridSearchCV (optimizing for '{primary_metric}')...")

        grid_search = GridSearchCV(model, param_grid, cv=inner_cv, n_jobs=-1, scoring=primary_metric)
        grid_search.fit(X_train_fold, y_train_fold)

        # Salva i migliori parametri per questo fold
        all_fold_best_params.append(grid_search.best_params_)
        if verbose:
            print(f"  Best Params for this fold: {grid_search.best_params_}")

        # --- 4. VALUTAZIONE SUL TEST SET ESTERNO ---
        best_model_for_fold = grid_search.best_estimator_
        y_pred = best_model_for_fold.predict(X_val)

        if verbose:
            print("  Calculating metrics on the outer test set...")

        # Calcola e salva tutte le metriche richieste
        for metric in scoring:
            if metric == 'accuracy':
                score = accuracy_score(y_val, y_pred)
            elif metric == 'roc_auc':
                try:
                    y_score = best_model_for_fold.predict_proba(X_val)[:, 1]
                    score = roc_auc_score(y_val, y_score)
                except (AttributeError, IndexError):
                    score = np.nan # Modello non ha predict_proba o è binario/monoclasse
            elif metric == 'f1':
                # Determina automaticamente se usare binary, macro o weighted
                n_classes = len(np.unique(y_train))
                if n_classes == 2:
                    score = f1_score(y_val, y_pred, average='binary')
                else:
                    score = f1_score(y_val, y_pred, average='macro')
            elif metric == 'precision':
                # Tra tutti i punti assegnati alla classe j, quanti appartengono alla vera classe dominante? (Chi c’è nella classe?)
                n_classes = len(np.unique(y_train))
                if n_classes == 2:
                    score = precision_score(y_val, y_pred, average='binary')
                else:
                    score = precision_score(y_val, y_pred, average='macro', zero_division=0)
            elif metric == 'recall':
                # "Tra tutti i punti della vera classe A, quanti sono stati correttamente classificati?" (Dove sono finiti i punti della classe?)
                n_classes = len(np.unique(y_train))
                if n_classes == 2:
                    score = recall_score(y_val, y_pred, average='binary')
                else:
                    score = recall_score(y_val, y_pred, average='macro', zero_division=0)
            elif metric == 'r2':
                score = r2_score(y_val, y_pred)
            elif metric == 'mae':
                score = mean_absolute_error(y_val, y_pred)
            elif metric == 'mse':
                score = mean_squared_error(y_val, y_pred)
            elif metric == 'rmse':
                score = root_mean_squared_error(y_val, y_pred)
            else:
                score = np.nan # Metrica non riconosciuta

            score_results[metric].append(score)
            if verbose:
                print(f"    {metric.upper()}: {score:.4f}")

    # --- 5. RIEPILOGO FINALE ---
    if verbose:
        print("\n--- Nested Cross-Validation Final Report ---")

    final_summary = {}
    for metric, scores in score_results.items():
        mean_score = np.nanmean(scores)
        std_score = np.nanstd(scores)
        final_summary[metric] = {
            'mean': mean_score,
            'std': std_score,
            'all_scores': scores
        }
        if verbose:
            print(f"Final {metric.upper()} estimate: {mean_score:.4f} ± {std_score:.4f}")

            end_time = time()
            print(f"--- Total time: {end_time - start_time:.2f} seconds ---")

    return {
        'performance_summary': final_summary,
        'all_fold_best_params': all_fold_best_params
    }

# Best manifold

In [2]:
from sklearn.metrics import silhouette_score, adjusted_mutual_info_score
from sklearn.model_selection import ParameterSampler

def best_manifold(X, y, model, param_grid, metric="ami", n_iter=3,
                                          random_state=42):
    best_score = -np.inf
    best_params = None

    # Crea un campionamento casuale di combinazioni di parametri dalla griglia param_grid. Questo è più efficiente rispetto a esplorare tutte le combinazioni.
    sampler = ParameterSampler(param_grid, n_iter=n_iter, random_state=random_state)

    # per ogni set di parametri generato dal sampler
    for params in sampler:
        try:
            # creo il modello usando i parametri specificati
            embedding = model(**params)
            # eseguo l'algoritmo di embedding per ottenere una nuova rappresentazione dei dati nel nuovo spazio (ad esempio, 2d o 3d), chiamata X_embedded
            X_embedded = embedding.fit_transform(X)

            # silhouette calcolato solo se specificato nei parametri del metodo e se l'output dell'embedding è di due dimensioni
            if metric == "silhouette" and X_embedded.shape[1] == 2:
                score = silhouette_score(X_embedded, y)

            elif metric == "ami":
                score = adjusted_mutual_info_score(y, np.argmax(X_embedded, axis=1))
            else:
                continue

            print(f"Executing function--Params: {params} => {metric.upper()} Score: {score:.4f}")

            if score > best_score:
                best_score = score
                best_params = params

        except Exception as e:
            print(f"Error with params {params}: {e}")

    return {"Best Params": best_params, "Best Score": best_score}

# Plot embedding

In [3]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # needed for 3D plotting


def plot_embedding(X_embedded, y, title):
    dim = X_embedded.shape[1]
    if dim not in [2, 3]:
        raise ValueError("Embedding dimension must be 2 or 3 for visualization.")

    plt.figure(figsize=(8, 6))
    if dim == 2:
        plt.scatter(X_embedded[:, 0], X_embedded[:, 1], c=y, cmap='viridis', alpha=0.7, edgecolor='k')
        plt.xlabel("Component 1")
        plt.ylabel("Component 2")
    else:  # 3D plot
        ax = plt.axes(projection='3d')
        sc = ax.scatter(X_embedded[:, 0], X_embedded[:, 1], X_embedded[:, 2],
                        c=y, cmap='viridis', alpha=0.7, edgecolor='k')
        ax.set_xlabel("Component 1")
        ax.set_ylabel("Component 2")
        ax.set_zlabel("Component 3")

    plt.title(title)
    plt.tight_layout()
    plt.show()

# Train final model from Nested Cross Validation

In [4]:
from collections import Counter
from sklearn.base import clone
import numpy as np
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, # classificazione
    r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error # regressione
)

def train_final_model_from_nested_cv(model,
                                     all_fold_best_params,
                                     X, y,
                                     score_per_fold: list[float] = None,
                                     strategy: str = 'most_frequent',
                                     X_test=None, y_test=None,
                                     scoring: list[str] = None,
                                     verbose=True):
    """
    Allena un modello finale usando i migliori iperparametri ottenuti da una nested CV,
    e calcola le metriche su un test set opzionale se fornito.

    Parameters:
        model: modello sklearn
        all_fold_best_params: lista dei parametri ottimali per ogni fold
        X, y: dati completi per l'addestramento
        score_per_fold: punteggi dei fold esterni, richiesto per 'best_fold'
        strategy: 'most_frequent' o 'best_fold'
        X_test, y_test: test set opzionale per calcolare le metriche finali
        scoring: lista di metriche da calcolare (default auto)
        verbose: se True, stampa info

    Returns:
        final_model: modello allenato su tutto il dataset
        best_params: iperparametri usati
        test_metrics
    """
    if not all_fold_best_params:
        raise ValueError("La lista di best_params è vuota.")

    if strategy == 'most_frequent':
        # Conta la combinazione più ricorrente tra i dizionari
        param_counts = Counter([frozenset(p.items()) for p in all_fold_best_params])
        most_common_params = dict(param_counts.most_common(1)[0][0])
        if verbose:
            print(f"\n[STRATEGIA: most_frequent] Parametri più frequenti sui fold:")
            print(most_common_params)
        best_params = most_common_params

    elif strategy == 'best_fold':
        if score_per_fold is None:
            raise ValueError("score_per_fold è richiesto per la strategia 'best_fold'.")
        if len(score_per_fold) != len(all_fold_best_params):
            raise ValueError("score_per_fold e all_fold_best_params devono avere la stessa lunghezza.")

        best_index = int(np.nanargmax(score_per_fold))
        best_params = all_fold_best_params[best_index]

        if verbose:
            print(f"\n[STRATEGIA: best_fold] Selezionato il fold #{best_index + 1} con punteggio migliore: {score_per_fold[best_index]:.4f}")
            print(f"Parametri selezionati: {best_params}")

    else:
        raise ValueError("Strategia non supportata: usa 'most_frequent' o 'best_fold'.")

    # Clona il modello e imposta i parametri selezionati
    final_model = clone(model).set_params(**best_params)

    # Allena su tutto il dataset
    final_model.fit(X, y)

    test_metrics = {}
    # Calcolo delle metriche su test set, se fornito
    if X_test is not None and y_test is not None:
        if verbose:
            print("\nCalcolo delle metriche sul test set finale...")

        y_pred = final_model.predict(X_test)
        try:
            y_proba = final_model.predict_proba(X_test)[:, 1]
        except:
            y_proba = None

        # Determina task
        est_type = getattr(final_model, "_estimator_type", None)
        is_clf = (est_type == "classifier")

        if scoring is None:
            scoring = ['accuracy', 'roc_auc'] if is_clf else ['r2', 'mae', 'rmse']

        for metric in scoring:
            if metric == 'accuracy':
                test_metrics['accuracy'] = accuracy_score(y_test, y_pred)
            elif metric == 'roc_auc':
                if y_proba is not None:
                    test_metrics['roc_auc'] = roc_auc_score(y_test, y_proba)
                else:
                    test_metrics['roc_auc'] = np.nan
            elif metric == 'f1':
                # Determina automaticamente se usare binary, macro o weighted
                n_classes = len(np.unique(y_train))
                if n_classes == 2:
                    test_metrics['f1'] = f1_score(y_val, y_pred, average='binary')
                else:
                    test_metrics['f1'] = f1_score(y_val, y_pred, average='macro')
            elif metric == 'precision':
                n_classes = len(np.unique(y_train))
                if n_classes == 2:
                    test_metrics['precision'] = precision_score(y_val, y_pred, average='binary')
                else:
                    test_metrics['precision'] = precision_score(y_val, y_pred, average='macro', zero_division=0)
            elif metric == 'recall':
                n_classes = len(np.unique(y_train))
                if n_classes == 2:
                    test_metrics['recall'] = recall_score(y_val, y_pred, average='binary')
                else:
                    test_metrics['recall'] = recall_score(y_val, y_pred, average='macro', zero_division=0)
            elif metric == 'r2':
                test_metrics['r2'] = r2_score(y_test, y_pred)
            elif metric == 'mae':
                test_metrics['mae'] = mean_absolute_error(y_test, y_pred)
            elif metric == 'rmse':
                test_metrics['rmse'] = root_mean_squared_error(y_test, y_pred)
            else:
                test_metrics[metric] = np.nan

        if verbose:
            for m, v in test_metrics.items():
                print(f"  {m.upper()}: {v:.4f}")

    return final_model, best_params, test_metrics


# Simple K-fold Cross Validation (not nested)

In [5]:
# cross_val_score calcola gli scores del modello usando la cv specificata
from sklearn.model_selection import KFold, cross_val_score
import numpy as np


def evaluate_with_kfold(model, X, y, n_splits=5):
    # verifica delle dimensioni per assicurarsi che il numero di campioni in X corrisponda al numero di etichette
    if len(X) != len(y):
        print(f"Error: Mismatched lengths - X: {len(X)}, y: {len(y)}")
        return None

    # definizione del k-fold
    # n_splits specifica il numero di fold
    # shuffle mischia i dati prima di suddividerli

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    try:
        # valutazione del modello in ogni modello in ognuno dei fold
        scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')

        # media e deviazione standard dei punteggi sui fold
        mean_acc = np.mean(scores)
        std_acc = np.std(scores)
        print(f"Mean Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
        return mean_acc, std_acc
    except ValueError as e:
        print(f"Error during cross-validation: {e}")
        return None